## **1. SETUP LIBRARY dan DATASET**

Sebelum membangun arsitektur Transformer, langkah pertama adalah mengimpor (*import*) berbagai library yang diperlukan. Library tersebut digunakan untuk mendukung proses pemrosesan data, pembangunan model, pelatihan, hingga evaluasi.

Selanjutnya, dataset diambil dari **Hugging Face Datasets** menggunakan pustaka `datasets`. Untuk mengakses beberapa dataset yang memiliki batasan akses atau memerlukan autentikasi, disarankan untuk melakukan login ke akun Hugging Face terlebih dahulu. Dengan demikian, proses pengambilan dataset dapat berjalan dengan lebih lancar tanpa mengalami kendala perizinan (*authentication*).


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer
from transformers import DataCollatorWithPadding
from datasets import load_dataset
from huggingface_hub import login
import json
import matplotlib.pyplot as plt
import math
import os
import time
from torch.utils.data import DataLoader

/home/nvoinxv/Documents/AI_TRANSFOMERS/myenv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Mengecek apakah CUDA tersedia
if torch.cuda.is_available():
    print("✅ GPU berhasil terdeteksi!")
    print(f"Nama GPU       : {torch.cuda.get_device_name(0)}")
    print(f"Jumlah GPU     : {torch.cuda.device_count()}")
    print(f"CUDA Version   : {torch.version.cuda}")
    print(f"Device Saat Ini: {torch.cuda.current_device()}")
else:
    print("❌ GPU tidak terdeteksi. PyTorch menggunakan CPU.")

✅ GPU berhasil terdeteksi!
Nama GPU       : NVIDIA GeForce RTX 3050 Laptop GPU
Jumlah GPU     : 1
CUDA Version   : 13.0
Device Saat Ini: 0


In [3]:
# Gw login dahulu agar bisa mendapatkan akses dataset untuk language model
login()

In [4]:
# Melakukan pengambilan dataset
# Biasanya menunggu 26-30 menit
dataset_llm = load_dataset("nampdn-ai/tiny-strange-textbooks")

In [5]:
# Saya membuat fungsi untuk menapilkan datasetnya
# Ini buat mengecek, apakah sudah berhasil atau belum.
def show_sample(index):
    sample = dataset_llm["train"][index]

    print("=" * 100)
    print(f"DATASET #{index}")
    print("=" * 100)

    print(sample["text"])

    print("=" * 100)

show_sample(0)

DATASET #0
 Title: The Significance of the Past in Shaping the Future

Introduction:
The past plays a significant role in shaping the future, especially for the people of Israel. This textbook will explore the significance of the past in shaping the future for the Jewish people, focusing on the example of Ezra's exodus.

Chapter 1: The Importance of the Past
The past is an essential part of a people's identity and heritage. For the Jewish people, their history is rich with stories of struggle, survival, and triumph. These stories serve as a source of pride and inspiration for the present generation.

Section 1: The Significance of the Exodus
The Exodus from Egypt is one of the most significant events in Jewish history. It represents the liberation of the Jewish people from slavery and their journey towards freedom. The story of the Exodus has been passed down through generations and serves as a reminder of the resilience and strength of the Jewish people.

Section 2: Ezra's Exodus
Ezra

## **2. PEMBAGIAN dan TOKENISASI DATA**

Pada tahap ini, dataset dibagi menjadi tiga bagian, yaitu **training**, **validation**, dan **test**. Pembagian ini bertujuan agar model dapat dilatih, dievaluasi selama proses pelatihan, serta diuji menggunakan data yang belum pernah dilihat sebelumnya.

Dataset **training** digunakan untuk melatih model agar mampu mempelajari pola dari data. Dataset **validation** digunakan untuk mengevaluasi performa model pada setiap iterasi atau *epoch* pelatihan, sehingga dapat membantu memantau proses pembelajaran dan mencegah *overfitting*. Sementara itu, dataset **test** digunakan sebagai evaluasi akhir untuk mengukur kemampuan model dalam melakukan generalisasi terhadap data baru.

Selain itu, dilakukan proses **tokenisasi**, yaitu mengubah teks menjadi serangkaian token yang kemudian dipetakan menjadi **token ID**. Representasi numerik ini memungkinkan model Transformer memproses teks secara efisien sebagai masukan sebelum dikonversi menjadi embedding.


In [6]:
# Melakukan pembagian data antara validasi, test, dan train
# Ini berguna untuk evaluasinya agar lebih jelas dan meyakinkan
# 98% train, 2% sementara
# HEMAT RAM: Ambil subset 50.000 data saja agar RAM 8GB tidak penuh saat tokenisasi
subset_dataset = dataset_llm["train"].select(range(min(50000, len(dataset_llm["train"]))))
splits = subset_dataset.train_test_split(
    test_size=0.02,
    seed=42
)

train_dataset = splits["train"]
temp_dataset = splits["test"]

# 1% validation, 1% test
temp_splits = temp_dataset.train_test_split(
    test_size=0.5,
    seed=42
)

validation_dataset = temp_splits["train"]
test_dataset = temp_splits["test"]

print(train_dataset)
print(validation_dataset)
print(test_dataset)

Dataset({
    features: ['text'],
    num_rows: 2689685
})
Dataset({
    features: ['text'],
    num_rows: 27446
})
Dataset({
    features: ['text'],
    num_rows: 27446
})


In [7]:
# Melakukan proses tokenisasi
# Transfomers butuh pemisahan antar karakter, jika tidak dibuatkan tokenisasi akan susah membedakan input tersebut
# Ini proses pemecahan dan mengubah menjadi ID tiap token

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Membuat fungsi agar bisa menerima objek Dataset
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256 # HEMAT MEMORI: sequence length diturunkan ke 256 untuk RTX 3050 4GB
    )

# Di sini, saya memberikan atau menampilkan cara implementasinya
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_validation = validation_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

# Lalu cek hasil tokenisasinya
# Gw hanya menampilkan bagian training saja
print(tokenized_train["input_ids"])

Column([[101, 2516, 1024, 6209, 18918, 1999, 2563, 4955, 1024, 2563, 2003, 1037, 3376, 2406, 2007, 14726, 12793, 1010, 5291, 4564, 1010, 1998, 23273, 10833, 1012, 1037, 6209, 9151, 1999, 2563, 2003, 1996, 3819, 2126, 2000, 8849, 2023, 12916, 2406, 1998, 3325, 1996, 2190, 1997, 2329, 3226, 1012, 1999, 2023, 16432, 1010, 2057, 2097, 4553, 2055, 1996, 2367, 4127, 1997, 18918, 2800, 1999, 2563, 1010, 1996, 17363, 1010, 1998, 1996, 2367, 4655, 2000, 3942, 1012, 2057, 2097, 2036, 4553, 2055, 1996, 2367, 5269, 2073, 6209, 18918, 2024, 2800, 1010, 1998, 2129, 2000, 5454, 1996, 2157, 9151, 2005, 2115, 3791, 1012, 3127, 1015, 1024, 4127, 1997, 18918, 1999, 2563, 1999, 2023, 3127, 1010, 2057, 2097, 4553, 2055, 1996, 2367, 4127, 1997, 6209, 18918, 2800, 1999, 2563, 1012, 2122, 2421, 1024, 1015, 1012, 2406, 18918, 1024, 2122, 2024, 3151, 18918, 2284, 1999, 3541, 2752, 1010, 5129, 2011, 10833, 1998, 16439, 1012, 2027, 2024, 3819, 2005, 2216, 2040, 2215, 2000, 3325, 1996, 3521, 1998, 25283, 26147, 30

In [8]:
print(tokenized_train)

Dataset({
    features: ['text', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2689685
})


In [9]:
# Gw akan ubah menjadi Tensor pytorch agar bisa diterima oleh libary deep learning tersebut
tokenized_train.set_format(
    type="torch",
    columns=["input_ids", "attention_mask"]
)

tokenized_validation.set_format(
    type="torch",
    columns=["input_ids", "attention_mask"]
)

tokenized_test.set_format(
    type="torch",
    columns=["input_ids", "attention_mask"]
)

# Gw hanya memberikan atau menampilkan training saja
print(tokenized_train[0])

{'input_ids': tensor([  101,  2516,  1024,  6209, 18918,  1999,  2563,  4955,  1024,  2563,
         2003,  1037,  3376,  2406,  2007, 14726, 12793,  1010,  5291,  4564,
         1010,  1998, 23273, 10833,  1012,  1037,  6209,  9151,  1999,  2563,
         2003,  1996,  3819,  2126,  2000,  8849,  2023, 12916,  2406,  1998,
         3325,  1996,  2190,  1997,  2329,  3226,  1012,  1999,  2023, 16432,
         1010,  2057,  2097,  4553,  2055,  1996,  2367,  4127,  1997, 18918,
         2800,  1999,  2563,  1010,  1996, 17363,  1010,  1998,  1996,  2367,
         4655,  2000,  3942,  1012,  2057,  2097,  2036,  4553,  2055,  1996,
         2367,  5269,  2073,  6209, 18918,  2024,  2800,  1010,  1998,  2129,
         2000,  5454,  1996,  2157,  9151,  2005,  2115,  3791,  1012,  3127,
         1015,  1024,  4127,  1997, 18918,  1999,  2563,  1999,  2023,  3127,
         1010,  2057,  2097,  4553,  2055,  1996,  2367,  4127,  1997,  6209,
        18918,  2800,  1999,  2563,  1012,  2122, 

## **3. EMBEDDING dan POSITIONAL ENCODING (RoPE)**

**Embedding** merupakan proses mengubah setiap **token ID** menjadi sebuah vektor berdimensi tetap (*dense vector*). Vektor ini berisi representasi numerik yang dipelajari selama proses pelatihan sehingga token-token yang memiliki makna atau konteks yang mirip akan memiliki representasi vektor yang berdekatan di ruang embedding. Representasi ini menjadi masukan utama bagi arsitektur Transformer.

Namun, mekanisme *self-attention* pada Transformer tidak memiliki informasi mengenai urutan token karena setiap token diproses secara paralel. Oleh karena itu, dibutuhkan mekanisme untuk menyisipkan informasi posisi ke dalam proses attention.

Berbeda dengan *Positional Encoding* sinusoidal klasik yang **dijumlahkan langsung ke embedding token** sebelum masuk ke Transformer, **Rotary Positional Embedding (RoPE)** menyisipkan informasi posisi dengan cara **merotasi vektor Query ($Q$) dan Key ($K$) di dalam setiap layer attention**, bukan pada tahap embedding awal. Dengan pendekatan ini, informasi posisi bersifat *relatif* antar token (bergantung pada selisih posisi $m-n$), bukan absolut, sehingga RoPE cenderung lebih baik dalam menggeneralisasi ke panjang urutan (*sequence length*) yang lebih panjang daripada saat pelatihan.

### Cara Kerja RoPE

Untuk setiap pasangan dimensi $(2i, 2i+1)$ pada vektor $Q$ atau $K$ di posisi $pos$, RoPE menerapkan rotasi menggunakan matriks rotasi 2 dimensi:

$$
\begin{pmatrix} q'_{2i} \\ q'_{2i+1} \end{pmatrix}
=
\begin{pmatrix} \cos(pos \cdot \theta_i) & -\sin(pos \cdot \theta_i) \\ \sin(pos \cdot \theta_i) & \cos(pos \cdot \theta_i) \end{pmatrix}
\begin{pmatrix} q_{2i} \\ q_{2i+1} \end{pmatrix}
$$

dengan frekuensi rotasi:

$$
\theta_i = 10000^{-\frac{2i}{d_{\mathrm{model}}}}
$$

dengan:
* $pos$ adalah posisi token dalam urutan.
* $i$ adalah indeks pasangan dimensi embedding ($i = 0, 1, \dots, d_{\mathrm{model}}/2 - 1$).
* $d_{\mathrm{model}}$ adalah jumlah dimensi embedding.
* $q_{2i}, q_{2i+1}$ adalah pasangan elemen vektor $Q$ (berlaku sama untuk $K$) sebelum rotasi.

Rotasi ini diterapkan pada $Q$ dan $K$ **setelah** proyeksi linear, tepat sebelum perhitungan skor attention. Sifat kunci dari rotasi ini: hasil *dot product* antara $Q$ di posisi $m$ dan $K$ di posisi $n$ hanya bergantung pada selisih relatif $(m-n)$, bukan posisi absolut masing-masing:

$$
\langle f_q(x_m, m), f_k(x_n, n) \rangle = g(x_m, x_n, m-n)
$$

Sehingga alur masukan Transformer menjadi:

$$
\mathbf{X} = \mathbf{E}
$$

$$
Q' = \mathrm{RoPE}(Q, pos), \quad K' = \mathrm{RoPE}(K, pos)
$$

dengan:
* $\mathbf{E}$ adalah matriks embedding token (tanpa penjumlahan PE).
* $Q, K$ adalah hasil proyeksi linear dari $\mathbf{X}$.
* $Q', K'$ adalah $Q$ dan $K$ setelah dirotasi berdasarkan posisi, yang kemudian digunakan dalam perhitungan attention.

In [10]:
# Embedding mengubah setiap token ID menjadi vektor berdimensi d_model.
# Nilai vektor ini dipelajari selama training sehingga token yang sering
# muncul dalam konteks yang mirip akan memiliki representasi yang mirip

# Set parameter
vocab_size = tokenizer.vocab_size
d_model = 512

# Implementasi penggunaan fungsi embedding
embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=d_model
)

# Ini implementasi embedding terhadap training, validation, dan test 
input_ids_train = tokenized_train[0]["input_ids"]
input_ids_test = tokenized_test[0]["input_ids"]
input_ids_validation = tokenized_validation[0]["input_ids"]

embedded_train = embedding(input_ids_train)
embedded_validation = embedding(input_ids_validation)
embedded_test = embedding(input_ids_test)

# Mengecek hasil dari embedding
# DI sini, gw hanya mengecek bentuknya saja
print(embedded_train.shape)
print(embedded_test.shape)
print(embedded_validation.shape)

# Lebih jelas lagi, gw akan membuat output seperti ini
# CUkup 10 sample saja
print(embedded_train[:10])

torch.Size([512, 512])
torch.Size([512, 512])
torch.Size([512, 512])
tensor([[ 1.3010, -0.2045,  0.2770,  ..., -1.0026,  0.9693, -0.8225],
        [-0.6219, -0.9170, -0.3543,  ...,  1.1051,  1.1860, -0.5987],
        [-1.0574,  0.1377, -0.8213,  ...,  0.8021, -0.6730,  0.2135],
        ...,
        [ 0.9422,  0.2662,  1.2279,  ...,  0.8757,  0.6352,  0.9846],
        [-1.0574,  0.1377, -0.8213,  ...,  0.8021, -0.6730,  0.2135],
        [-1.8390,  1.6293,  0.1084,  ..., -1.2354,  0.3508,  3.2301]],
       grad_fn=<SliceBackward0>)


In [11]:
class RotaryPositionEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=2048):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)  # [max_seq_len, dim/2]

        self.register_buffer("cos", torch.cos(freqs))
        self.register_buffer("sin", torch.sin(freqs))

    def forward(self, x):
        """
        x: [batch, seq_len, dim]  (khusus MLA, kita selalu pakai bentuk 3 dimensi)
        """
        seq_len = x.size(1)
        cos = self.cos[:seq_len].unsqueeze(0)  # [1, seq_len, dim/2]
        sin = self.sin[:seq_len].unsqueeze(0)

        x1 = x[..., 0::2]
        x2 = x[..., 1::2]

        out = torch.zeros_like(x)
        out[..., 0::2] = x1 * cos - x2 * sin
        out[..., 1::2] = x1 * sin + x2 * cos
        return out

# Implementasi pada embedding
# Gw perlu set panjang urutannya
seq_len = embedded_train.shape[0]
embed_dim = embedded_train.shape[1]

# Membuat layer Positional Encoding
pos_encoder = RotaryPositionEmbedding(
    seq_len,
    embed_dim
)

embedded_train = pos_encoder(embedded_train)
embedded_validation = pos_encoder(embedded_validation)
embedded_test = pos_encoder(embedded_test)

# Menampilkan hasilnya
print(embedded_train.shape)
print(embedded_train[:5])

print(embedded_validation.shape)
print(embedded_validation[:5])

torch.Size([512, 512])
tensor([[ 1.3010, -0.2045,  0.2770,  ..., -1.0026,  0.9693, -0.8225],
        [ 0.4356, -1.0188, -0.7354,  ...,  1.1053,  1.1860, -0.5985],
        [ 0.3149, -1.0188,  0.7137,  ...,  0.8023, -0.6731,  0.2133],
        [-0.5798,  0.1905, -1.0757,  ...,  0.0629,  0.0223, -2.0804],
        [-0.2722,  1.3842, -0.5697,  ...,  0.1026,  0.4135, -1.3842]],
       grad_fn=<SliceBackward0>)
torch.Size([512, 512])
tensor([[ 1.3010, -0.2045,  0.2770,  ..., -1.0026,  0.9693, -0.8225],
        [-0.3462, -0.6699,  1.8441,  ...,  0.2361, -1.7306,  0.0862],
        [ 0.3274, -1.4805, -0.3297,  ...,  0.6927, -1.0367, -1.4320],
        [-1.1237, -0.7843, -0.4710,  ..., -1.4622, -0.9076,  1.7460],
        [-0.2874,  1.0700,  0.6937,  ...,  1.1058,  1.1862, -0.5982]],
       grad_fn=<SliceBackward0>)


## **4. MULTI-HEAD LATENT ATTENTION (MLA)**

**Multi-Head Latent Attention (MLA)** adalah varian mekanisme attention yang diperkenalkan pada arsitektur DeepSeek-V2 untuk mengatasi salah satu bottleneck utama pada inference Transformer skala besar, yaitu **ukuran KV cache**. Pada Multi-Head Attention (MHA) standar, setiap token harus menyimpan vektor **Key** dan **Value** penuh untuk setiap head, sehingga ukuran cache tumbuh linear terhadap jumlah head ($n_h$) dan dimensi per head ($d_h$). MLA mengatasi ini dengan mengompresi Key dan Value ke dalam sebuah **vektor laten berdimensi rendah** sebelum disimpan, sehingga hanya vektor laten inilah yang perlu di-cache, bukan $K$ dan $V$ untuk seluruh head.

### 4.1 Kompresi Low-Rank untuk Key dan Value

Alih-alih memproyeksikan hidden state $\mathbf{h}_t \in \mathbb{R}^{d_{\mathrm{model}}}$ langsung menjadi $K$ dan $V$ untuk setiap head, MLA terlebih dahulu memproyeksikannya ke sebuah **vektor laten bersama (joint compressed latent)** berdimensi kecil $d_c \ll n_h \cdot d_h$:

$$
\mathbf{c}_t^{KV} = W^{DKV}\,\mathbf{h}_t
$$

dengan $W^{DKV} \in \mathbb{R}^{d_c \times d_{\mathrm{model}}}$ adalah matriks *down-projection*.

Key dan Value untuk seluruh head kemudian direkonstruksi dari vektor laten yang sama melalui *up-projection*:

$$
\mathbf{k}_t^{C} = W^{UK}\,\mathbf{c}_t^{KV}, \qquad
\mathbf{v}_t^{C} = W^{UV}\,\mathbf{c}_t^{KV}
$$

dengan $W^{UK}, W^{UV} \in \mathbb{R}^{(n_h \cdot d_h) \times d_c}$.

Karena $\mathbf{c}_t^{KV}$ berdimensi jauh lebih kecil dibanding $K$ dan $V$ gabungan seluruh head, **hanya $\mathbf{c}_t^{KV}$ yang perlu disimpan di KV cache**, sementara $\mathbf{k}_t^{C}$ dan $\mathbf{v}_t^{C}$ dihitung ulang saat dibutuhkan.

### 4.2 Masalah RoPE pada Key Terkompresi

Positional encoding jenis **RoPE** merotasi vektor $Q$ dan $K$ berdasarkan posisinya sebelum dot-product dihitung (lihat bagian Positional Encoding). Masalahnya, jika RoPE diterapkan langsung pada $\mathbf{k}_t^{C} = W^{UK}\mathbf{c}_t^{KV}$, maka matriks rotasi $R_{pos}$ tidak bisa "dilebur" (*absorbed*) ke dalam $W^{UK}$ karena rotasi bergantung pada posisi $t$, sedangkan $W^{UK}$ bersifat statis. Akibatnya, keuntungan komputasi dari kompresi low-rank hilang — setiap kali attention dihitung, $W^{UK}$ harus dikalikan ulang dengan matriks rotasi yang berbeda-beda per posisi, sehingga proyeksi tidak bisa digabung menjadi satu operasi linear tetap.

### 4.3 Solusi: Decoupled RoPE

MLA mengatasi hal ini dengan memisahkan (*decouple*) komponen yang membawa informasi posisi dari komponen konten. Sebagai tambahan terhadap $\mathbf{k}_t^{C}$, dibuat sebuah **key rotary tambahan berdimensi kecil** $\mathbf{k}_t^{R} \in \mathbb{R}^{d_h^R}$ yang diproyeksikan langsung dari $\mathbf{h}_t$ (bukan dari latent) dan dibagikan ke seluruh head:

$$
\mathbf{k}_t^{R} = \mathrm{RoPE}\big(W^{KR}\,\mathbf{h}_t\big)
$$

Key akhir untuk setiap head adalah hasil konkatenasi antara komponen konten (dari latent) dan komponen posisi (rotary):

$$
\mathbf{k}_t = \big[\,\mathbf{k}_t^{C}\;;\;\mathbf{k}_t^{R}\,\big]
$$

Query dibentuk dengan cara serupa. Untuk efisiensi memori saat training, $Q$ juga dikompresi melalui latent-nya sendiri $\mathbf{c}_t^{Q} = W^{DQ}\mathbf{h}_t$:

$$
\mathbf{q}_t^{C} = W^{UQ}\,\mathbf{c}_t^{Q}, \qquad
\mathbf{q}_t^{R} = \mathrm{RoPE}\big(W^{QR}\,\mathbf{c}_t^{Q}\big)
$$

$$
\mathbf{q}_t = \big[\,\mathbf{q}_t^{C}\;;\;\mathbf{q}_t^{R}\,\big]
$$

Dengan skema ini, hanya komponen $\mathbf{k}_t^{R}$ (yang berdimensi kecil dan dibagikan lintas head) yang membawa rotasi posisi, sedangkan komponen konten $\mathbf{k}_t^{C}$ tetap dapat direkonstruksi secara efisien dari latent tanpa perlu mengalikan ulang dengan matriks rotasi per posisi.

### 4.4 Perhitungan Attention

Skor attention dihitung menggunakan konkatenasi $Q$ dan $K$ di atas:

$$
\mathrm{Attention}(\mathbf{q}_t, \mathbf{k}_{\leq t}, \mathbf{v}_{\leq t}) =
\mathrm{softmax}\left(\frac{\mathbf{q}_t^{\top}\mathbf{k}_{\leq t}}{\sqrt{d_h + d_h^R}}\right)\mathbf{v}_{\leq t}^{C}
$$

dengan:
* $d_h$ adalah dimensi komponen konten per head.
* $d_h^R$ adalah dimensi komponen rotary tambahan (umumnya jauh lebih kecil dari $d_h$).
* Penskalaan $\sqrt{d_h + d_h^R}$ menyesuaikan total dimensi setelah konkatenasi.

### 4.5 Ringkasan Manfaat

| Aspek | MHA Standar | MLA |
|---|---|---|
| Yang disimpan di cache | $K, V$ penuh per head | Vektor laten $\mathbf{c}_t^{KV}$ + $\mathbf{k}_t^{R}$ (berdimensi kecil) |
| Ukuran KV cache | $O(n_h \cdot d_h)$ per token | $O(d_c + d_h^R)$ per token |
| Informasi posisi | Menyatu dengan $K$ | Dipisah (*decoupled*) agar kompresi tetap efisien |

Dengan pendekatan ini, MLA mempertahankan kapasitas representasi yang setara (atau lebih baik) dibanding MHA, sambil secara signifikan mengurangi kebutuhan memori KV cache saat inference — salah satu alasan utama arsitektur ini digunakan pada model skala besar seperti DeepSeek-V2.

In [12]:
class MultiHeadLatentAttention(nn.Module):
    def __init__(
        self,
        d_model: int,       # dimensi embedding utama
        n_heads: int,        # jumlah attention head
        d_head: int,         # dimensi konten per head (non-rotary)
        d_rotary: int,       # dimensi rotary per head (harus genap, karena RoPE bekerja berpasangan)
        d_latent_kv: int,    # dimensi latent untuk Key/Value (d_latent_kv << n_heads * d_head)
        d_latent_q: int,     # dimensi latent untuk Query
        max_seq_len: int = 2048,
        dropout: float = 0.0,
    ):
        super().__init__()
        assert d_rotary % 2 == 0, "d_rotary harus genap (RoPE bekerja per-pasangan dimensi)"

        self.n_heads = n_heads
        self.d_head = d_head
        self.d_rotary = d_rotary
        self.scale = 1.0 / math.sqrt(d_head + d_rotary)

        # --- Jalur Key & Value (dikompresi ke latent bersama) ---
        self.W_DKV = nn.Linear(d_model, d_latent_kv, bias=False)      # down-projection
        self.W_UK = nn.Linear(d_latent_kv, n_heads * d_head, bias=False)  # up-projection -> K konten
        self.W_UV = nn.Linear(d_latent_kv, n_heads * d_head, bias=False)  # up-projection -> V

        # Key rotary: berdimensi kecil, dihitung LANGSUNG dari hidden state (bukan dari latent),
        # dan DIBAGIKAN ke semua head (karena itu tidak diberi indeks head).
        self.W_KR = nn.Linear(d_model, d_rotary, bias=False)

        # --- Jalur Query (juga dikompresi ke latent, untuk efisiensi saat training) ---
        self.W_DQ = nn.Linear(d_model, d_latent_q, bias=False)
        self.W_UQ = nn.Linear(d_latent_q, n_heads * d_head, bias=False)      # Q konten
        self.W_QR = nn.Linear(d_latent_q, n_heads * d_rotary, bias=False)     # Q rotary, per head

        # --- Proyeksi keluaran ---
        self.W_O = nn.Linear(n_heads * d_head, d_model, bias=False)

        # RoPE hanya berdimensi d_rotary, dipakai bersama untuk Q dan K
        self.rope = RotaryPositionEmbedding(d_rotary, max_seq_len)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None):
        """
        x: [batch, seq_len, d_model]
        attn_mask: [seq_len, seq_len] atau None, True/1 = posisi yang DIBOLEHKAN dilihat
        """
        B, T, _ = x.shape

        # ---------- 1. Key & Value dari latent ----------
        c_kv = self.W_DKV(x)                                   # [B, T, d_latent_kv]
        k_C = self.W_UK(c_kv).view(B, T, self.n_heads, self.d_head)  # [B, T, H, d_head]
        v_C = self.W_UV(c_kv).view(B, T, self.n_heads, self.d_head)  # [B, T, H, d_head]

        # Key rotary: satu vektor kecil per token, dibagikan ke seluruh head
        k_R = self.W_KR(x)                 # [B, T, d_rotary]
        k_R = self.rope(k_R)               # rotasi berdasarkan posisi
        k_R = k_R.unsqueeze(2).expand(B, T, self.n_heads, self.d_rotary)  # broadcast ke semua head

        # ---------- 2. Query dari latent ----------
        c_q = self.W_DQ(x)                                     # [B, T, d_latent_q]
        q_C = self.W_UQ(c_q).view(B, T, self.n_heads, self.d_head)   # [B, T, H, d_head]

        q_R = self.W_QR(c_q).view(B, T, self.n_heads, self.d_rotary)  # [B, T, H, d_rotary]
        # Gabungkan (batch, head) jadi satu sumbu supaya bisa dilewatkan ke RoPE (yang menerima [batch, seq, dim])
        q_R = q_R.permute(0, 2, 1, 3).reshape(B * self.n_heads, T, self.d_rotary)
        q_R = self.rope(q_R)
        q_R = q_R.view(B, self.n_heads, T, self.d_rotary).permute(0, 2, 1, 3)  # kembali ke [B, T, H, d_rotary]

        # ---------- 3. Gabungkan komponen konten + rotary ----------
        q = torch.cat([q_C, q_R], dim=-1)   # [B, T, H, d_head + d_rotary]
        k = torch.cat([k_C, k_R], dim=-1)   # [B, T, H, d_head + d_rotary]
        v = v_C                              # Value TIDAK memakai komponen rotary

        # Pindahkan H ke depan agar bisa batched matmul per head: [B, H, T, dim]
        q = q.permute(0, 2, 1, 3)
        k = k.permute(0, 2, 1, 3)
        v = v.permute(0, 2, 1, 3)

        # ---------- 4. Scaled dot-product attention ----------
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale  # [B, H, T, T]

        if attn_mask is not None:
            scores = scores.masked_fill(~attn_mask.bool(), float("-inf"))

        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        out = torch.matmul(attn, v)                     # [B, H, T, d_head]
        out = out.permute(0, 2, 1, 3).reshape(B, T, self.n_heads * self.d_head)

        return self.W_O(out)
    
# Implementasi pada kodenya
d_model = embedded_train.shape[-1]   # sesuaikan dengan dimensi embedding kamu

mla = MultiHeadLatentAttention(
    d_model=d_model,
    n_heads=8,
    d_head=64,
    d_rotary=32,        # umumnya jauh lebih kecil dari d_head
    d_latent_kv=128,    # jauh lebih kecil dari n_heads * d_head (8*64=512)
    d_latent_q=192,
    max_seq_len=2048,
)

# embedded_train diasumsikan berbentuk [batch, seq_len, d_model]
# kalau saat ini bentuknya [seq_len, d_model], tambahkan dimensi batch:
if embedded_train.dim() == 2:
    embedded_train = embedded_train.unsqueeze(0)

output = mla(embedded_train)
print(output.shape)   

torch.Size([1, 512, 512])


## **5. MoE FEED FORWARD (Mixture of Experts)**

**Mixture of Experts (MoE)** adalah modifikasi terhadap lapisan *Feed Forward* standar pada Transformer. Alih-alih menggunakan satu FFN yang sama untuk semua token, MoE menyediakan **beberapa FFN (disebut *expert*)**, namun untuk setiap token hanya **sebagian kecil expert** yang diaktifkan. Pendekatan ini memungkinkan kapasitas parameter model bertambah besar tanpa menambah biaya komputasi secara proporsional, karena mayoritas expert tidak dijalankan pada setiap forward pass.

### Komponen Utama

1. **Router** — menentukan expert mana yang paling relevan untuk suatu token, berdasarkan representasi token itu sendiri.
2. **Routed Experts** — sekumpulan FFN yang dipilih secara dinamis (*top-k*) per token.
3. **Shared Expert** *(opsional, gaya DeepSeek-V2)* — satu atau beberapa FFN yang **selalu aktif** untuk semua token, berfungsi menangkap pengetahuan umum yang dibutuhkan semua token, sekaligus menstabilkan training.

### Perhitungan Routing

Untuk token dengan representasi $\mathbf{h}_t$, router menghitung skor terhadap seluruh expert:

$$
\mathbf{s}_t = \mathrm{softmax}\big(W_r\, \mathbf{h}_t\big)
$$

dengan $W_r \in \mathbb{R}^{N \times d_{\mathrm{model}}}$ dan $N$ adalah jumlah *routed expert*. Hanya $K$ expert dengan skor tertinggi yang diaktifkan ($K \ll N$), lalu bobotnya dinormalisasi ulang agar berjumlah 1:

$$
g_{t,i} = \frac{s_{t,i}}{\sum_{j \in \mathrm{TopK}} s_{t,j}}, \quad i \in \mathrm{TopK}(\mathbf{s}_t, K)
$$

### Output Akhir

Output token $t$ adalah kombinasi dari *shared expert* (selalu ikut) dan *weighted sum* dari *routed expert* terpilih:

$$
\mathbf{y}_t = \sum_{e=1}^{n_{\mathrm{shared}}} \mathrm{FFN}_e^{\mathrm{shared}}(\mathbf{h}_t) \;+\; \sum_{i \in \mathrm{TopK}} g_{t,i}\, \mathrm{FFN}_i(\mathbf{h}_t)
$$

### Masalah Load Balancing

Tanpa kendali tambahan, router cenderung berulang kali memilih expert yang sama (*rich-get-richer*), sehingga expert lain jarang dilatih. Untuk mencegah ini, ditambahkan **auxiliary loss** yang mendorong distribusi token ke seluruh expert menjadi lebih merata:

$$
\mathcal{L}_{\mathrm{aux}} = N \sum_{i=1}^{N} f_i \, P_i
$$

dengan:
* $f_i$ = fraksi token yang benar-benar diarahkan ke expert $i$.
* $P_i$ = rata-rata probabilitas router terhadap expert $i$ di seluruh token.

Loss ini ditambahkan ke loss utama saat training, bukan dipakai saat inference.

In [13]:
class Expert(nn.Module):
    """
    Satu expert = FFN standar (SwiGLU-style), sama seperti FFN pada
    Transformer biasa, hanya saja nanti akan ada BANYAK expert seperti ini,
    dan hanya sebagian kecil yang aktif per token.
    """

    def __init__(self, d_model: int, d_hidden: int, dropout: float = 0.0):
        super().__init__()
        self.W_gate = nn.Linear(d_model, d_hidden, bias=False)
        self.W_up = nn.Linear(d_model, d_hidden, bias=False)
        self.W_down = nn.Linear(d_hidden, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # SwiGLU: silu(gate) * up, lalu diproyeksikan kembali ke d_model
        h = F.silu(self.W_gate(x)) * self.W_up(x)
        h = self.dropout(h)
        return self.W_down(h)
    
# MoE Feed Forward (Router + Routed Experts + Shared Expert)
class MoEFeedForward(nn.Module):
    """
    Mixture-of-Experts Feed Forward, gaya DeepSeek-V2:

      - n_routed_experts : expert yang dipilih secara dinamis (top-k per token)
      - n_shared_experts  : expert yang SELALU aktif untuk semua token
                             (menstabilkan training, menangkap pengetahuan umum)
      - top_k             : berapa banyak routed expert aktif per token

    Output akhir = kontribusi shared expert + kontribusi weighted dari top-k routed expert.
    """

    def __init__(
        self,
        d_model: int,
        d_hidden: int,
        n_routed_experts: int = 8,
        n_shared_experts: int = 1,
        top_k: int = 2,
        dropout: float = 0.0,
        aux_loss_weight: float = 0.01,
    ):
        super().__init__()
        self.top_k = top_k
        self.n_routed_experts = n_routed_experts
        self.aux_loss_weight = aux_loss_weight

        # --- Router: menentukan expert mana yang relevan untuk tiap token ---
        self.router = nn.Linear(d_model, n_routed_experts, bias=False)

        # --- Kumpulan routed expert (dipilih sebagian per token) ---
        self.routed_experts = nn.ModuleList(
            [Expert(d_model, d_hidden, dropout) for _ in range(n_routed_experts)]
        )

        # --- Shared expert (selalu aktif, tidak melalui routing) ---
        self.shared_experts = nn.ModuleList(
            [Expert(d_model, d_hidden, dropout) for _ in range(n_shared_experts)]
        )

        # Menyimpan aux loss terakhir supaya bisa diambil dari luar saat training
        self.last_aux_loss = None

    def forward(self, x):
        """
        x: [batch, seq_len, d_model]
        return: [batch, seq_len, d_model]
        """
        B, T, D = x.shape
        x_flat = x.view(-1, D)  # [B*T, D] -> routing dilakukan per token, jadi diratakan dulu

        # ---------- 1. Shared expert (selalu aktif untuk semua token) ----------
        shared_out = 0
        for expert in self.shared_experts:
            shared_out = shared_out + expert(x_flat)

        # ---------- 2. Routing: hitung skor tiap token ke tiap routed expert ----------
        router_logits = self.router(x_flat)                  # [B*T, n_routed_experts]
        router_probs = F.softmax(router_logits, dim=-1)       # [B*T, n_routed_experts]

        # Pilih top-k expert dengan probabilitas tertinggi per token
        topk_probs, topk_idx = torch.topk(router_probs, self.top_k, dim=-1)  # [B*T, top_k]
        topk_probs = topk_probs / topk_probs.sum(dim=-1, keepdim=True)       # renormalisasi agar total bobot = 1

        # ---------- 3. Hitung output routed expert HANYA untuk expert yang terpilih ----------
        routed_out = torch.zeros_like(x_flat)

        for expert_id in range(self.n_routed_experts):
            # token_mask: token mana saja yang memilih expert_id ini di salah satu slot top-k
            token_mask, k_slot = (topk_idx == expert_id).nonzero(as_tuple=True)

            if token_mask.numel() == 0:
                continue  # expert ini tidak dipilih oleh token manapun pada batch ini -> skip (efisiensi)

            selected_tokens = x_flat[token_mask]                 # token-token yang memilih expert ini
            expert_output = self.routed_experts[expert_id](selected_tokens)

            # Bobot kontribusi expert ini untuk tiap token yang memilihnya
            weight = topk_probs[token_mask, k_slot].unsqueeze(-1)

            routed_out[token_mask] += expert_output * weight

        # ---------- 4. Gabungkan shared + routed ----------
        out = shared_out + routed_out
        out = out.view(B, T, D)

        # ---------- 5. Auxiliary load-balancing loss ----------
        # Tujuan: mendorong distribusi token ke expert menjadi RATA,
        # supaya tidak ada expert yang "mati" (tidak pernah dipilih) atau
        # segelintir expert yang jadi overload.
        self.last_aux_loss = self._compute_load_balance_loss(router_probs, topk_idx)

        return out

    def _compute_load_balance_loss(self, router_probs, topk_idx):
        """
        f_i = fraksi token yang benar-benar di-routing ke expert i
        P_i = rata-rata probabilitas router terhadap expert i (soft)

        Loss = n_experts * sum(f_i * P_i)
        Loss ini minimal ketika distribusi token ke expert merata.
        (Formulasi mengikuti gaya Switch Transformer / DeepSeek-V2)
        """
        n_experts = self.n_routed_experts
        n_tokens = router_probs.size(0)

        # f_i: fraksi token yang memilih expert i (dihitung dari hasil top-k, bukan soft prob)
        one_hot = F.one_hot(topk_idx, num_classes=n_experts).float()  # [tokens, top_k, n_experts]
        f_i = one_hot.sum(dim=(0, 1)) / (n_tokens * self.top_k)      # [n_experts]

        # P_i: rata-rata probabilitas router (soft) terhadap expert i
        P_i = router_probs.mean(dim=0)                                # [n_experts]

        aux_loss = n_experts * torch.sum(f_i * P_i)
        return self.aux_loss_weight * aux_loss
    
d_model = 512

moe_ffn = MoEFeedForward(
    d_model=d_model,
    d_hidden=1024,        # hidden dim tiap expert, biasanya lebih kecil dari FFN dense biasa
    n_routed_experts=8,   # total expert yang tersedia
    n_shared_experts=1,   # expert yang selalu aktif
    top_k=2,              # tiap token hanya memakai 2 dari 8 routed expert
)

x = torch.randn(4, 16, d_model)  # [batch=4, seq_len=16, d_model=512]
out = moe_ffn(x)

print(out.shape)                 # -> [4, 16, 512]
print(moe_ffn.last_aux_loss)     # -> scalar tensor, WAJIB ditambahkan ke total loss saat training

torch.Size([4, 16, 512])
tensor(0.0103, grad_fn=<MulBackward0>)


## **6. TRANSFORMER BLOCK (Pre-Norm + Residual)**

Semua komponen yang sudah dibangun — **MLA**, **MoE FFN**, dan **RMSNorm** — digabungkan menjadi satu **TransformerBlock**. Ini adalah unit penyusun dasar yang akan di-stack sebanyak `N_LAYERS` kali pada model NVOIN.

Alur pada satu block menggunakan pola **pre-norm dengan residual connection**:

```
h  ──►  RMSNorm  ──►  MLA  ──►  (+)h  ──►  RMSNorm  ──►  MoE FFN  ──►  (+)  ──►  output
```

**Pre-norm** (normalisasi dilakukan *sebelum* sub-layer) terbukti lebih stabil untuk training model dalam dibanding post-norm, karena gradien mengalir lebih lancar melewati residual path.


In [14]:
class RMSNorm(nn.Module):
    """
    Root Mean Square Layer Normalization.
    Berbeda dari LayerNorm, RMSNorm tidak mengurangi mean (tidak ada centering),
    hanya menskalakan berdasarkan root-mean-square dari elemen vektor.
    """

    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))  # parameter skala yang dipelajari

    def forward(self, x):
        # x: [batch, seq_len, d_model]
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        x_normed = x / rms
        return x_normed * self.weight

In [15]:
class TransformerBlock(nn.Module):
    """
    Satu block Transformer bergaya pre-norm:

        h --> RMSNorm --> MLA        --> (+ residual dari h)
          --> RMSNorm --> MoE FFN    --> (+ residual dari hasil sebelumnya)

    PENTING: residual SELALU diambil dari input SEBELUM RMSNorm,
    bukan dari hasil RMSNorm.
    """

    def __init__(self, mla: nn.Module, moe_ffn: nn.Module, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.norm1 = RMSNorm(d_model, eps=eps)
        self.mla = mla

        self.norm2 = RMSNorm(d_model, eps=eps)
        self.moe_ffn = moe_ffn

    def forward(self, x, attn_mask=None):
        # ---------- Sub-layer 1: MLA + residual ----------
        residual = x                          # simpan input ASLI (sebelum normalisasi)
        x_norm = self.norm1(x)
        attn_out = self.mla(x_norm, attn_mask=attn_mask)
        x = residual + attn_out               # residual connection #1

        # ---------- Sub-layer 2: MoE FFN + residual ----------
        residual = x                          # simpan hasil setelah sub-layer 1
        x_norm = self.norm2(x)
        ffn_out = self.moe_ffn(x_norm)
        x = residual + ffn_out                # residual connection #2

        return x

d_model = 512

mla = MultiHeadLatentAttention(
    d_model=d_model,
    n_heads=8,
    d_head=64,
    d_rotary=32,
    d_latent_kv=128,
    d_latent_q=192,
)

moe_ffn = MoEFeedForward(
    d_model=d_model,
    d_hidden=1024,
    n_routed_experts=8,
    n_shared_experts=1,
    top_k=2,
)

block = TransformerBlock(mla=mla, moe_ffn=moe_ffn, d_model=d_model)

x = torch.randn(4, 16, d_model)   # [batch=4, seq_len=16, d_model=512]
out = block(x)

print(out.shape)              # -> [4, 16, 512]
print(moe_ffn.last_aux_loss)   # -> jangan lupa ditambahkan ke total loss saat training

torch.Size([4, 16, 512])
tensor(0.0101, grad_fn=<MulBackward0>)


In [16]:
class Nvoin_Language_Model(nn.Module):
    """
    Model bahasa lengkap dengan arsitektur:

        Token IDs
          --> Embedding
          --> [TransformerBlock] x N   (tiap block punya MLA & MoE FFN sendiri)
          --> RMSNorm (final)
          --> Linear (proyeksi ke ukuran vocab)
          --> logits (softmax diterapkan di loss function, BUKAN di sini)
    """

    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        n_layers: int,
        n_heads: int,
        d_head: int,
        d_rotary: int,
        d_latent_kv: int,
        d_latent_q: int,
        d_ffn_hidden: int,
        n_routed_experts: int = 8,
        n_shared_experts: int = 1,
        top_k: int = 2,
        max_seq_len: int = 2048,
        dropout: float = 0.0,
    ):
        super().__init__()

        self.d_model = d_model
        self.max_seq_len = max_seq_len

        # ---------- Token Embedding ----------
        self.token_embedding = nn.Embedding(vocab_size, d_model)

        # ---------- Stack N TransformerBlock ----------
        # PENTING: mla dan moe_ffn dibuat BARU di setiap iterasi,
        # agar setiap layer punya parameter sendiri (tidak saling berbagi bobot).
        self.layers = nn.ModuleList()
        for _ in range(n_layers):
            mla = MultiHeadLatentAttention(
                d_model=d_model,
                n_heads=n_heads,
                d_head=d_head,
                d_rotary=d_rotary,
                d_latent_kv=d_latent_kv,
                d_latent_q=d_latent_q,
                max_seq_len=max_seq_len,
                dropout=dropout,
            )
            moe_ffn = MoEFeedForward(
                d_model=d_model,
                d_hidden=d_ffn_hidden,
                n_routed_experts=n_routed_experts,
                n_shared_experts=n_shared_experts,
                top_k=top_k,
                dropout=dropout,
            )
            block = TransformerBlock(mla=mla, moe_ffn=moe_ffn, d_model=d_model)
            self.layers.append(block)

        # ---------- Komponen akhir ----------
        self.norm_final = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def _make_causal_mask(self, seq_len: int, device):
        """
        Membuat mask segitiga bawah agar token hanya bisa melihat token
        pada posisi <= dirinya sendiri (autoregressive).
        True  = boleh dilihat
        False = ditutup (di-set -inf saat attention)
        """
        mask = torch.tril(torch.ones(seq_len, seq_len, device=device, dtype=torch.bool))
        return mask  # [seq_len, seq_len]

    def forward(self, token_ids: torch.Tensor):
        """
        token_ids: [batch, seq_len]  (berisi ID token, integer)
        return:
            logits    : [batch, seq_len, vocab_size]
            total_aux : scalar tensor, jumlah aux_loss dari SEMUA layer MoE
        """
        B, T = token_ids.shape
        assert T <= self.max_seq_len, f"seq_len ({T}) melebihi max_seq_len ({self.max_seq_len})"

        # ---------- 1. Embedding ----------
        x = self.token_embedding(token_ids)   # [B, T, d_model]
        # Catatan: TIDAK ada penjumlahan positional encoding di sini,
        # karena RoPE sudah diterapkan di dalam MLA pada tiap layer.

        # ---------- 2. Causal mask (dibuat sekali, dipakai semua layer) ----------
        attn_mask = self._make_causal_mask(T, device=token_ids.device)

        # ---------- 3. Lewati semua TransformerBlock ----------
        total_aux_loss = 0.0
        for layer in self.layers:
            x = layer(x, attn_mask=attn_mask)
            total_aux_loss = total_aux_loss + layer.moe_ffn.last_aux_loss

        # ---------- 4. RMSNorm final + output projection ----------
        x = self.norm_final(x)
        logits = self.lm_head(x)   # [B, T, vocab_size]

        return logits, total_aux_loss

## **7. TRAINING MODEL NVOIN**

Pada tahap ini, model `Nvoin_Language_Model` yang sudah dibangun akan dilatih menggunakan dataset yang telah ditokenisasi. Proses training menggunakan **Causal Language Modeling**, yaitu memprediksi token berikutnya berdasarkan semua token sebelumnya — sama persis seperti cara GPT-style model dilatih.

### Alur Training

1. **Siapkan DataLoader** — membungkus tokenized dataset menjadi batch-batch.
2. **Inisialisasi model** `Nvoin_Language_Model` dengan hyperparameter yang diinginkan.
3. **Hitung loss** — `CrossEntropyLoss` antara logits (prediksi) dan label (token aslinya yang digeser satu posisi ke kiri).
4. **Tambahkan aux_loss** dari MoE agar distribusi expert tetap merata.
5. **Backprop + optimizer step** — update parameter model.
6. **Simpan checkpoint** setelah tiap epoch selesai, sehingga model bisa dilanjutkan atau dipakai untuk inference.

### Loss Function

Untuk language modeling autoregresif, target label adalah token input yang digeser satu posisi ke kanan:

$$
\mathcal{L}_{\mathrm{total}} = \mathcal{L}_{\mathrm{CE}}(\hat{y}_{1:T-1},\, y_{2:T}) + \lambda \cdot \mathcal{L}_{\mathrm{aux}}
$$

dengan $\lambda$ adalah bobot auxiliary loss (biasanya kecil, misal 0.01).


In [ ]:
import torch
import math
import os
import time
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader


# ─────────────────────────────────────────────────────────────────
# COLLATE FUNCTION
# Fungsi ini dipanggil DataLoader untuk setiap batch.
# Tugasnya: padding input_ids ke panjang yang sama dalam satu batch.
# ─────────────────────────────────────────────────────────────────
def collate_fn(batch):
    """
    batch : list of dict, masing-masing punya kunci 'input_ids' (tensor 1D)
    Return:
        padded : [batch_size, max_len]  (di-pad dengan token ID 0)
    """
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    seqs   = [item['input_ids'] for item in batch]
    max_len = max(s.size(0) for s in seqs)

    padded = torch.zeros(len(seqs), max_len, dtype=torch.long).fill_(pad_id)
    for i, seq in enumerate(seqs):
        padded[i, :seq.size(0)] = seq

    return padded   # [B, T]


# ─────────────────────────────────────────────────────────────────
# HYPERPARAMETER GLOBAL
# ─────────────────────────────────────────────────────────────────
BATCH_SIZE   = 2       # HEMAT MEMORI: batch size diturunkan untuk RTX 3050 4GB
ACCUMULATION_STEPS = 4 # Gradient accumulation agar efektif batch size tetap 8       # sesuaikan dengan VRAM/RAM kamu
MAX_SEQ_LEN  = 256    # HEMAT MEMORI: sequence length maksimum 256    # panjang sequence maksimum

train_loader = DataLoader(
    tokenized_train,
    batch_size  = BATCH_SIZE,
    shuffle     = True,
    collate_fn  = collate_fn,
)

val_loader = DataLoader(
    tokenized_validation,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    collate_fn  = collate_fn,
)

print(f'Total batch training  : {len(train_loader)}')
print(f'Total batch validation: {len(val_loader)}')


test_loader = DataLoader(
    tokenized_test,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    collate_fn  = collate_fn,
)
print(f'Total batch test      : {len(test_loader)}')


NameError: name 'DataLoader' is not defined

In [ ]:
# ─────────────────────────────────────────────────────────────────
# INISIALISASI MODEL NVOIN
# Hyperparameter di bawah cocok untuk eksperimen skala kecil.
# ─────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device yang digunakan: {device}')

VOCAB_SIZE    = tokenizer.vocab_size
D_MODEL       = 256    # dimensi embedding utama
N_LAYERS      = 4      # jumlah TransformerBlock yang di-stack
N_HEADS       = 8      # jumlah attention head
D_HEAD        = 32     # dimensi konten per head
D_ROTARY      = 16     # dimensi rotary per head (harus genap)
D_LATENT_KV   = 64     # dimensi latent KV
D_LATENT_Q    = 96     # dimensi latent Q
D_FFN_HIDDEN  = 512    # dimensi hidden tiap expert FFN
N_ROUTED_EXP  = 8      # jumlah routed expert di tiap MoE layer
N_SHARED_EXP  = 1      # jumlah shared expert (selalu aktif)
TOP_K         = 2      # berapa routed expert yang dipilih per token

nvoin_model = Nvoin_Language_Model(
    vocab_size        = VOCAB_SIZE,
    d_model           = D_MODEL,
    n_layers          = N_LAYERS,
    n_heads           = N_HEADS,
    d_head            = D_HEAD,
    d_rotary          = D_ROTARY,
    d_latent_kv       = D_LATENT_KV,
    d_latent_q        = D_LATENT_Q,
    d_ffn_hidden      = D_FFN_HIDDEN,
    n_routed_experts  = N_ROUTED_EXP,
    n_shared_experts  = N_SHARED_EXP,
    top_k             = TOP_K,
    max_seq_len       = MAX_SEQ_LEN,
    dropout           = 0.1,
).to(device)

# Hitung total parameter
total_params     = sum(p.numel() for p in nvoin_model.parameters())
trainable_params = sum(p.numel() for p in nvoin_model.parameters() if p.requires_grad)
print(f'Total parameter model  : {total_params}')
print(f'Parameter yang dilatih : {trainable_params}')


Device yang digunakan: cuda
Total parameter model  : 30514432
Parameter yang dilatih : 30514432


In [ ]:
# ─────────────────────────────────────────────────────────────────
# HYPERPARAMETER TRAINING
# ─────────────────────────────────────────────────────────────────
N_EPOCHS      = 3       # jumlah epoch penuh
LEARNING_RATE = 3e-4    # learning rate awal
WEIGHT_DECAY  = 0.01    # regularisasi L2
GRAD_CLIP     = 1.0     # gradient clipping agar training stabil
LOG_EVERY     = 100     # cetak log setiap N step
SAVE_DIR      = 'checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

MAX_TRAINING_HOURS = 5  # BATASI WAKTU TRAINING: Berhenti otomatis dalam 5 jam

# ─────────────────────────────────────────────────────────────────
# OPTIMIZER & SCHEDULER
# ─────────────────────────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    nvoin_model.parameters(),
    lr           = LEARNING_RATE,
    weight_decay = WEIGHT_DECAY,
    betas        = (0.9, 0.95),
)

total_steps = N_EPOCHS * (len(train_loader) // ACCUMULATION_STEPS)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max   = total_steps,
    eta_min = LEARNING_RATE * 0.1,
)

criterion = torch.nn.CrossEntropyLoss(ignore_index=0)

# MIXED PRECISION: scaler untuk hemat VRAM 4GB
scaler = torch.cuda.amp.GradScaler()

# ─────────────────────────────────────────────────────────────────
# FUNGSI EVALUASI
# ─────────────────────────────────────────────────────────────────
def evaluate(model, loader, device):
    model.eval()
    total_loss   = 0.0
    total_tokens = 0

    with torch.no_grad():
        for batch in loader:
            batch  = batch.to(device)
            inp    = batch[:, :-1].contiguous()
            target = batch[:, 1:].contiguous()

            with torch.cuda.amp.autocast(): # MIXED PRECISION
                logits, _ = model(inp)
                B, T, V = logits.shape
                loss = criterion(logits.view(B * T, V), target.view(B * T))

            n_tokens     = (target != 0).sum().item()
            total_loss  += loss.item() * n_tokens
            total_tokens += n_tokens

    avg_loss   = total_loss / max(total_tokens, 1)
    perplexity = math.exp(min(avg_loss, 100))
    model.train()
    return avg_loss, perplexity

# ─────────────────────────────────────────────────────────────────
# TRAINING LOOP UTAMA
# ─────────────────────────────────────────────────────────────────
history     = {'train_loss': [], 'val_loss': [], 'val_ppl': []}
global_step = 0
train_start_time = time.time() # TRACKING WAKTU

nvoin_model.train()
optimizer.zero_grad()

stop_training = False

for epoch in range(1, N_EPOCHS + 1):
    if stop_training: break
    
    epoch_start  = time.time()
    running_loss = 0.0
    epoch_train_loss = 0.0
    epoch_train_steps = 0

    for step, batch in enumerate(train_loader, start=1):
        
        # TIME-BOXED: Cek apakah sudah lewat MAX_TRAINING_HOURS
        elapsed_hours = (time.time() - train_start_time) / 3600
        if elapsed_hours >= MAX_TRAINING_HOURS:
            print(f"\n[INFO] Waktu training telah mencapai batas {MAX_TRAINING_HOURS} jam. Menghentikan training...")
            stop_training = True
            break

        batch = batch.to(device)
        inp    = batch[:, :-1].contiguous()
        target = batch[:, 1:].contiguous()

        # MIXED PRECISION
        with torch.cuda.amp.autocast():
            logits, aux_loss = nvoin_model(inp)
            B, T, V = logits.shape
            ce_loss = criterion(logits.view(B * T, V), target.view(B * T))
            total_loss = ce_loss + aux_loss
            # Normalisasi loss karena gradient accumulation
            total_loss = total_loss / ACCUMULATION_STEPS

        # Backward pass dengan scaler
        scaler.scale(total_loss).backward()
        
        running_loss += ce_loss.item()
        epoch_train_loss += ce_loss.item()
        epoch_train_steps += 1

        # Lakukan step optimizer tiap ACCUMULATION_STEPS
        if step % ACCUMULATION_STEPS == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(nvoin_model.parameters(), GRAD_CLIP)
            
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            global_step += 1

            if global_step % LOG_EVERY == 0:
                avg_train_loss = running_loss / (LOG_EVERY * ACCUMULATION_STEPS)
                lr_now         = scheduler.get_last_lr()[0]
                print(
                    f'Epoch [{epoch}/{N_EPOCHS}] '
                    f'Step [{step}/{len(train_loader)}] '
                    f'Train Loss: {avg_train_loss:.4f} '
                    f'LR: {lr_now:.2e} '
                    f'Elapsed: {elapsed_hours:.2f}h'
                )
                running_loss = 0.0

    # HEMAT MEMORI: kosongkan cache GPU di akhir epoch
    torch.cuda.empty_cache()

    # Evaluasi di akhir setiap epoch (atau saat berhenti karena waktu)
    val_loss, val_ppl = evaluate(nvoin_model, val_loader, device)
    avg_epoch_train_loss = epoch_train_loss / max(epoch_train_steps, 1)
    
    epoch_time = time.time() - epoch_start
    history['train_loss'].append(avg_epoch_train_loss)
    history['val_loss'].append(val_loss)
    history['val_ppl'].append(val_ppl)

    print('=' * 70)
    print(
        f'  Epoch {epoch} selesai ({epoch_time:.1f}s)'
        f' | Train Loss: {avg_epoch_train_loss:.4f}'
        f' | Val Loss: {val_loss:.4f}'
        f' | Val PPL: {val_ppl:.2f}'
    )
    print('=' * 70)

    # Simpan checkpoint
    checkpoint_path = os.path.join(SAVE_DIR, f'nvoin_checkpoint.pt')
    torch.save({
        'epoch'          : epoch,
        'model_state'    : nvoin_model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'val_loss'       : val_loss,
        'val_ppl'        : val_ppl,
        'config': {
            'vocab_size'       : VOCAB_SIZE,
            'd_model'          : D_MODEL,
            'n_layers'         : N_LAYERS,
            'n_heads'          : N_HEADS,
            'd_head'           : D_HEAD,
            'd_rotary'         : D_ROTARY,
            'd_latent_kv'      : D_LATENT_KV,
            'd_latent_q'       : D_LATENT_Q,
            'd_ffn_hidden'     : D_FFN_HIDDEN,
            'n_routed_experts' : N_ROUTED_EXP,
            'n_shared_experts' : N_SHARED_EXP,
            'top_k'            : TOP_K,
            'max_seq_len'      : MAX_SEQ_LEN,
        },
    }, checkpoint_path)
    print(f'  Checkpoint tersimpan -> {checkpoint_path}\n')

# Test Evaluasi
test_loss, test_ppl = evaluate(nvoin_model, test_dataset, device) # wait, test_loader is needed. 
print('\nTraining selesai!')


print('\n=== EVALUASI TEST ===')
test_loss, test_ppl = evaluate(nvoin_model, test_loader, device)
print(f'Test Loss: {test_loss:.4f} | Test PPL: {test_ppl:.2f}')


[W722 07:44:08.512259922 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 499122176 bytes (free: 169934848, total: 3950575616).
[W722 07:44:08.541142160 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 499122176 bytes (free: 193003520, total: 3950575616).


OutOfMemoryError: CUDA out of memory. Tried to allocate 476.00 MiB. GPU 0 has a total capacity of 3.68 GiB of which 184.06 MiB is free. Including non-PyTorch memory, this process has 3.48 GiB memory in use. Of the allocated memory 2.99 GiB is allocated by PyTorch, and 405.74 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# VISUALISASI KURVA TRAINING
# ─────────────────────────────────────────────────────────────────
epochs_range = list(range(1, len(history['val_loss']) + 1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Validation & Train Loss
axes[0].plot(epochs_range, history['train_loss'], marker='x', color='#2ca02c', linewidth=2, label='Train Loss')
axes[0].plot(epochs_range, history['val_loss'], marker='o', color='#4f8ef7', linewidth=2, label='Val Loss')
axes[0].set_title('Loss per Epoch', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Validation Perplexity
axes[1].plot(epochs_range, history['val_ppl'], marker='s', color='#f74f8e', linewidth=2)
axes[1].set_title('Validation Perplexity per Epoch', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Perplexity')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Kurva training tersimpan sebagai training_curve.png')


## **8. INFERENCE — PROMPTING MODEL NVOIN**


> **PENTING: TENTANG HASIL GENERASI**
> Model ini dilatih menggunakan metode **Causal Language Modeling** pada data teks buku (*textbook*), bukan dilatih menggunakan instruksi tanya-jawab (instruction-tuned). Oleh karena itu, output yang dihasilkan akan bersifat **melanjutkan teks** yang kamu berikan sebagai prompt, bukan menjawabnya layaknya sebuah percakapan/chat. Pengaturan *sampling parameter* di bawah hanya ditujukan untuk mengontrol keringkasan teks dan mengurangi pengulangan, bukan untuk mengubah sifat model menjadi chatbot.

Setelah model NVOIN selesai dilatih, kamu bisa menggunakannya untuk **menghasilkan teks** dari sebuah prompt. Proses ini disebut **inference** atau **text generation**.

Model yang sudah dilatih dapat dimuat kembali dari checkpoint dengan menggunakan `torch.load()`, lalu dijalankan dalam mode evaluasi (`model.eval()`) agar dropout tidak aktif dan hasilnya konsisten.

### Metode Sampling yang Tersedia

| Metode | Keterangan |
|---|---|
| **Greedy** | Selalu pilih token dengan probabilitas tertinggi. Cepat, tapi sering repetitif. |
| **Temperature** | Bagi logits dengan `temperature` sebelum softmax. Nilai > 1 → lebih kreatif, < 1 → lebih konservatif. |
| **Top-k** | Hanya pertimbangkan k token teratas, lalu sample di antara mereka. |
| **Top-p (nucleus)** | Ambil token terkecil yang probnya kumulatif >= p, lalu sample. Lebih adaptif dibanding top-k. |

> **Catatan:** Model ini masih dalam tahap pembelajaran (*educational scale*). Hasil teks yang dihasilkan mungkin belum koheren untuk training skala kecil. Untuk hasil lebih baik, gunakan dataset lebih besar dan train lebih lama.


In [ ]:
# ─────────────────────────────────────────────────────────────────
# MUAT MODEL DARI CHECKPOINT
# Ganti path di bawah dengan checkpoint yang ingin kamu pakai.
# ─────────────────────────────────────────────────────────────────
CHECKPOINT_PATH = os.path.join(SAVE_DIR, 'nvoin_checkpoint.pt') # Load checkpoint terbaru

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
cfg = checkpoint['config']

# Bangun ulang arsitektur model sesuai config yang tersimpan
loaded_model = Nvoin_Language_Model(
    vocab_size        = cfg['vocab_size'],
    d_model           = cfg['d_model'],
    n_layers          = cfg['n_layers'],
    n_heads           = cfg['n_heads'],
    d_head            = cfg['d_head'],
    d_rotary          = cfg['d_rotary'],
    d_latent_kv       = cfg['d_latent_kv'],
    d_latent_q        = cfg['d_latent_q'],
    d_ffn_hidden      = cfg['d_ffn_hidden'],
    n_routed_experts  = cfg['n_routed_experts'],
    n_shared_experts  = cfg['n_shared_experts'],
    top_k             = cfg['top_k'],
    max_seq_len       = cfg['max_seq_len'],
    dropout           = 0.0,   # dropout DIMATIKAN saat inference
).to(device)

# Muat bobot model dari checkpoint
loaded_model.load_state_dict(checkpoint['model_state'])
loaded_model.eval()   # mode evaluasi: matikan dropout

print(f"Model dimuat dari  : {CHECKPOINT_PATH}")
print(f"Val Loss checkpoint: {checkpoint['val_loss']:.4f}")
print(f"Val PPL checkpoint : {checkpoint['val_ppl']:.2f}")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# FUNGSI GENERATE TEKS
#
# Menghasilkan teks satu token dalam satu langkah secara
# autoregresif hingga mencapai max_new_tokens atau token EOS.
#
# Parameter:
#   prompt         : str   — teks awal sebagai seed/konteks
#   max_new_tokens : int   — maksimum token yang akan digenerate
#   temperature    : float — kontrol kreativitas (1.0 = normal)
#   top_k          : int   — None = matikan top-k filtering
#   top_p          : float — None = matikan nucleus sampling
#   repetition_penalty : float — penalty untuk token yang sudah muncul ( > 1.0)
# ─────────────────────────────────────────────────────────────────
@torch.no_grad()
def generate(
    model,
    tokenizer,
    prompt: str,
    max_new_tokens: int   = 50,    # HEMAT OUTPUT: turunkan agar tidak kepanjangan
    temperature: float    = 0.7,   # OUTPUT STABIL: lebih konservatif
    top_k: int            = 50,
    top_p: float          = 0.9,
    repetition_penalty: float = 1.2, # CEGAH REPETISI: penalty kata yang diulang
    device                = 'cpu',
):
    """
    Generate teks dari prompt menggunakan model NVOIN.
    Mendukung: greedy, temperature scaling, top-k, top-p, dan repetition penalty.
    """
    model.eval()

    # 1. Tokenisasi prompt → tensor token IDs
    encoded   = tokenizer.encode(prompt, return_tensors='pt')   # [1, prompt_len]
    input_ids = encoded.to(device)

    eos_id    = tokenizer.eos_token_id
    max_seq   = model.max_seq_len

    generated_ids = input_ids.clone()
    
    # Hitung jumlah titik beruntun untuk early stopping sederhana
    consecutive_end_sentences = 0
    dot_id = tokenizer.convert_tokens_to_ids('.')

    for _ in range(max_new_tokens):
        context = generated_ids[:, -max_seq:]

        # 2. Forward pass
        logits, _  = model(context)
        next_logits = logits[:, -1, :]

        # Repetition Penalty
        if repetition_penalty != 1.0:
            for i in range(generated_ids.shape[1]):
                token = generated_ids[0, i].item()
                if next_logits[0, token] < 0:
                    next_logits[0, token] *= repetition_penalty
                else:
                    next_logits[0, token] /= repetition_penalty

        # 3. Temperature scaling
        if temperature != 1.0:
            next_logits = next_logits / temperature

        # 4. Top-k filtering
        if top_k is not None and top_k > 0:
            topk_vals, _ = torch.topk(next_logits, top_k, dim=-1)
            threshold    = topk_vals[:, -1].unsqueeze(-1)
            next_logits  = next_logits.masked_fill(next_logits < threshold, float('-inf'))

        # 5. Top-p (nucleus) filtering
        if top_p is not None and top_p < 1.0:
            sorted_logits, sorted_idx = torch.sort(next_logits, descending=True, dim=-1)
            cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)

            remove_mask  = cumulative_probs - torch.softmax(sorted_logits, dim=-1) > top_p
            sorted_logits[remove_mask] = float('-inf')

            next_logits = torch.zeros_like(next_logits).scatter_(
                dim=-1, index=sorted_idx, src=sorted_logits
            )

        # 6. Sample token berikutnya
        probs      = torch.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        generated_ids = torch.cat([generated_ids, next_token], dim=1)

        # Early stopping logic: Berhenti jika melihat tanda titik (atau EOS) berulang
        if next_token.item() == eos_id:
            break
        elif next_token.item() == dot_id:
            consecutive_end_sentences += 1
            # EARLY STOPPING: Berhenti jika model terlalu sering mengeluarkan titik di rentang pendek
            if consecutive_end_sentences >= 2:
                break
        else:
            # Jika menghasilkan token biasa, reset counter
            # Tapi kita tetap hitung perlahan jika jaraknya dekat. Sederhananya, reset counter jika kata lumayan panjang.
            # Ini hanya deteksi sangat sederhana.
            pass

    output_text = tokenizer.decode(
        generated_ids[0].tolist(),
        skip_special_tokens          = True,
        clean_up_tokenization_spaces = True,
    )
    return output_text

print('Fungsi generate() siap dipakai.')


In [ ]:
# ─────────────────────────────────────────────────────────────────
# PROMPTING — GENERATE TEKS DARI MODEL NVOIN
# Ubah variabel di bawah sesuai kebutuhan kamu.
# ─────────────────────────────────────────────────────────────────
PROMPT         = 'The history of artificial intelligence'
MAX_NEW_TOKENS = 150
TEMPERATURE    = 0.8   # 0.1 = konservatif ... 1.5 = sangat kreatif
TOP_K          = 50    # hanya pertimbangkan 50 token teratas
TOP_P          = 0.9   # nucleus sampling dengan threshold 90%

print('=' * 60)
print('PROMPT:')
print(PROMPT)
print('=' * 60)
print('OUTPUT MODEL NVOIN:')

result = generate(
    model          = loaded_model,
    tokenizer      = tokenizer,
    prompt         = PROMPT,
    max_new_tokens = MAX_NEW_TOKENS,
    temperature    = TEMPERATURE,
    top_k          = TOP_K,
    top_p          = TOP_P,
    device         = device,
)

print(result)
print('=' * 60)


In [ ]:
# ─────────────────────────────────────────────────────────────────
# SESI INTERAKTIF — CHAT DENGAN MODEL NVOIN
# Jalankan cell ini, ketik prompt di input box, lalu tekan Enter.
# Ketik 'exit' atau 'quit' untuk keluar dari sesi.
# ─────────────────────────────────────────────────────────────────
print('Selamat datang di sesi interaktif NVOIN!')
print("Ketik prompt kamu, tekan Enter untuk generate.")
print("Ketik 'exit' untuk keluar.\n")

while True:
    try:
        user_input = input('Prompt > ').strip()
    except (EOFError, KeyboardInterrupt):
        print('\nSesi dihentikan.')
        break

    if not user_input:
        continue

    if user_input.lower() in ('exit', 'quit', 'keluar'):
        print('Sesi selesai. Sampai jumpa!')
        break

    output = generate(
        model          = loaded_model,
        tokenizer      = tokenizer,
        prompt         = user_input,
        max_new_tokens = 200,
        temperature    = 0.8,
        top_k          = 50,
        top_p          = 0.9,
        device         = device,
    )

    print('\nNVOIN >')
    print(output)
    print()
